# Flight Data Preprocessing with AWS Athena

This notebook processes raw flight data stored in S3 using AWS Athena and `awswrangler`. It defines database configurations, handles raw table mounting, builds a dynamic cleaning query, and executes CTAS statements to save cleaned data back to S3.

In [ ]:
import os
import warnings
import boto3
import awswrangler as wr
import re

# Suppress noisy Ray memory and future deprecation warnings
warnings.filterwarnings("ignore")
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"
os.environ["RAY_DISABLE_MEMORY_MONITOR"] = "1"

# --- Configuration ---
raw_bucket = "aai-540-group1-flight-data-raw"
processed_bucket = "aai-540-group1-flight-data-processed"
raw_prefix = "bts/2025/"

athena_db = "aai540_group1"
raw_table = "bts_raw_2025"

full_table_name = "clean_flights_full"
partitioned_table_name = "clean_flights_partitioned"

base_processed_path = f"s3://{processed_bucket}/cleaned/"
full_s3_path = f"{base_processed_path}full/"
partitioned_s3_path = f"{base_processed_path}partitioned/"

# Get AWS Account ID and Region dynamically for user-specific Athena results bucket
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = boto3.session.Session().region_name or "us-east-1"

# Explicit Athena query results location to fix awswrangler UserWarnings and isolate by user
athena_s3_output = f"s3://aws-athena-query-results-{account_id}-{region}/"

## 1. Bypass 403 & Get Headers

Here we fetch the file URIs to bypass prefix permissions, create the database if it doesn't exist, and extract the exact headers from a single chunk of the first file.

In [ ]:
print("Fetching file URIs to bypass prefix permissions...")
s3 = boto3.client("s3", region_name="us-east-1")
result = s3.list_objects_v2(Bucket=raw_bucket, Prefix=raw_prefix)
file_uris = [
    f"s3://{raw_bucket}/{item['Key']}"
    for item in result.get("Contents", [])
    if item["Key"].endswith(".csv.gz")
]

print("Creating database and extracting schema from source...")
wr.catalog.create_database(athena_db, exist_ok=True)

# Read just 1 chunk of the first file to get the exact headers
df_head = wr.s3.read_csv(path=[file_uris[0]], chunksize=1)
header = next(df_head).columns.tolist()

def normalize_header(name, index):
    normalized = re.sub(r"[^a-z0-9]", "", name.lower())
    if not normalized:
        normalized = f"unnamed{index}"
    if normalized[0].isdigit():
        normalized = "col" + normalized
    return normalized

columns = [normalize_header(name, i) for i, name in enumerate(header)]

## 2. Mount Raw Table & Show Original State

We generate the DDL to mount the external table in Athena pointing directly to our raw S3 files, then query it to display a sample of the data.

In [ ]:
print(f"Mounting raw Athena table: {athena_db}.{raw_table}...")
column_ddl = ",\n".join(f"  `{name}` string" for name in columns)
create_raw_table_sql = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS `{athena_db}`.`{raw_table}` ({column_ddl})
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES ('separatorChar'=',', 'quoteChar'='"')
STORED AS TEXTFILE
LOCATION 's3://{raw_bucket}/{raw_prefix}'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
wr.athena.start_query_execution(sql=create_raw_table_sql, database=athena_db, s3_output=athena_s3_output, wait=True)

print("\n=== ORIGINAL DATA SAMPLE ===")
df_raw_sample = wr.athena.read_sql_query(f'SELECT * FROM "{raw_table}" LIMIT 5', database=athena_db, s3_output=athena_s3_output)
display(df_raw_sample)
print("============================\n")

## 3. Dynamic SQL Builder for Cleaning

Here we construct a Create Table As Select (CTAS) query dynamically to clean and format the dataset.

**Preprocessing Steps & Rationale:**
1. **Drop Cancelled & Diverted Flights:** We filter out and drop `cancelled`, `cancellationcode`, and `diverted` columns. Downstream models will predict delays for *completed* flights, so non-completed flights are irrelevant to our target.
2. **Drop Multi-stage Diversion Data:** Columns starting with `div` (e.g., `div1airport`) track complex routing for diverted flights. Since we exclude diverted flights entirely, these sparse columns are removed to save space and reduce noise.
3. **Impute Missing Delay Reasons:** Delay columns (like `carrierdelay` or `weatherdelay`) are typically null in the raw data if the flight arrived on time. We cast these to `DOUBLE` and use `COALESCE` to replace nulls with `0.0` so numerical aggregations and machine learning algorithms work correctly.
4. **Enforce Target Variables:** We filter out any rows where critical target labels (`arrdel15`, `depdel15`) are missing.
5. **Format for Partitioning:**  Partitioning by month optimizes future queries by heavily reducing the amount of data scanned.

In [ ]:
print("Constructing CTAS transformation query...")

columns_to_drop = ['cancelled', 'cancellationcode', 'diverted']
delay_columns = ['carrierdelay', 'weatherdelay', 'nasdelay', 'securitydelay', 'lateaircraftdelay']

select_statements = []
partition_col = 'month'

for col in columns:
    if col in columns_to_drop or (col.startswith('div') and col != 'diverted'):
        continue

    if col == partition_col:
        continue

    if col in delay_columns:
        select_statements.append(f"COALESCE(TRY_CAST(NULLIF(TRIM({col}), '') AS DOUBLE), 0.0) AS {col}")
    else:
        select_statements.append(f"NULLIF(TRIM({col}), '') AS {col}")

select_statements.append(f"NULLIF(TRIM({partition_col}), '') AS {partition_col}")
select_clause = ",\n    ".join(select_statements)

where_clause = """
    TRY_CAST(NULLIF(TRIM(cancelled), '') AS DOUBLE) = 0
    AND TRY_CAST(NULLIF(TRIM(diverted), '') AS DOUBLE) = 0
    AND NULLIF(TRIM(arrdel15), '') IS NOT NULL
    AND NULLIF(TRIM(depdel15), '') IS NOT NULL
"""

## 4. Idempotent & Cross-Account CTAS Execution

We execute the CTAS queries to create the final cleaned tables (both full and partitioned).

**Why this approach?**
This step is designed to be **idempotent and cross-account safe**. It first checks if the target S3 path already contains data (e.g., if a teammate already processed it in their account).
* **If data exists:** It simply registers the existing Parquet files into your account's Glue Catalog using AWS Wrangler's metadata functions.
* **If data is missing:** It runs the full Athena CTAS query to process the raw data and write the new Parquet files to S3.

This prevents redundant data processing, saves time, and completely avoids S3 overwrite errors across different AWS accounts.

In [ ]:
print("Checking catalog and S3 for existing processed tables...")

# --- Process Full Table ---
if not wr.catalog.does_table_exist(database=athena_db, table=full_table_name):
    existing_full_files = wr.s3.list_objects(full_s3_path)
    if existing_full_files:
        print(f"\nData found at {full_s3_path}. Registering existing data to catalog...")
        wr.s3.store_parquet_metadata(
            path=full_s3_path,
            database=athena_db,
            table=full_table_name,
            dataset=True
        )
        print("✅ Full table registered successfully.")
    else:
        print(f"\nExecuting CTAS for Full Table -> {full_s3_path} (This may take a minute)...")
        ctas_full_query = f"""
        CREATE TABLE {full_table_name}
        WITH (
            format = 'PARQUET',
            external_location = '{full_s3_path}',
            parquet_compression = 'SNAPPY'
        ) AS
        SELECT
        {select_clause}
        FROM {raw_table}
        WHERE {where_clause}
        """
        wr.athena.start_query_execution(sql=ctas_full_query, database=athena_db, s3_output=athena_s3_output, wait=True)
        print("✅ Full table created successfully.")
else:
    print(f"SKIP: Table '{full_table_name}' already exists in catalog.")

# --- Process Partitioned Table ---
if not wr.catalog.does_table_exist(database=athena_db, table=partitioned_table_name):
    existing_part_files = wr.s3.list_objects(partitioned_s3_path)
    if existing_part_files:
        print(f"\nData found at {partitioned_s3_path}. Registering existing partitioned data to catalog...")
        wr.s3.store_parquet_metadata(
            path=partitioned_s3_path,
            database=athena_db,
            table=partitioned_table_name,
            dataset=True
        )
        print("✅ Partitioned table registered successfully.")
    else:
        print(f"\nExecuting CTAS for Partitioned Table -> {partitioned_s3_path} (This may take a minute)...")
        ctas_part_query = f"""
        CREATE TABLE {partitioned_table_name}
        WITH (
            format = 'PARQUET',
            external_location = '{partitioned_s3_path}',
            parquet_compression = 'SNAPPY',
            partitioned_by = ARRAY['{partition_col}']
        ) AS
        SELECT
        {select_clause}
        FROM {raw_table}
        WHERE {where_clause}
        """
        wr.athena.start_query_execution(sql=ctas_part_query, database=athena_db, s3_output=athena_s3_output, wait=True)
        print("✅ Partitioned table created successfully.")
else:
    print(f"SKIP: Table '{partitioned_table_name}' already exists in catalog.")

## 5. Final Report & Cleaned State

Compare the record counts before and after the cleaning logic, then sample the newly formatted dataset.

In [ ]:
if wr.catalog.does_table_exist(database=athena_db, table=full_table_name):
    print("\n=== CLEANING SUMMARY ===")
    raw_count = wr.athena.read_sql_query(f"SELECT COUNT(*) as count FROM {raw_table}", database=athena_db, s3_output=athena_s3_output)
    clean_count = wr.athena.read_sql_query(f"SELECT COUNT(*) as count FROM {full_table_name}", database=athena_db, s3_output=athena_s3_output)

    print(f"Initial Raw Rows:  {raw_count['count'][0]:,}")
    print(f"Final Clean Rows:  {clean_count['count'][0]:,}")
    print(f"Rows Removed:      {raw_count['count'][0] - clean_count['count'][0]:,}")
    print("=========================\n")

    print("=== CLEANED DATA SAMPLE ===")
    df_clean_sample = wr.athena.read_sql_query(f'SELECT * FROM "{full_table_name}" LIMIT 5', database=athena_db, s3_output=athena_s3_output)
    display(df_clean_sample)
    print("===========================\n")